# Run the experiments on Colab (GPU)

This notebook runs `src/train.py` on a Colab GPU and saves everything to Google Drive,
so nothing is lost if the runtime disconnects.

**Before you start:** `Runtime → Change runtime type → T4 GPU`.

**What you need in Google Drive:** the file `mahmutbey-traffic-forecast.zip` in `MyDrive/`.

**Expected time:** test stage ~1 hour, walk-forward stage ~1 hour on a T4.
If the runtime stops, just run all cells again: finished runs are skipped.

## 1 · Connect Drive and check the GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2 · Unpack the project

In [ ]:
import os
ZIP='/content/drive/MyDrive/mahmutbey-traffic-forecast.zip'
OUT='/content/drive/MyDrive/mahmutbey_results'          # results are written here, on Drive

!rm -rf /content/mahmutbey-traffic-forecast
!unzip -q $ZIP -d /content/
os.chdir('/content/mahmutbey-traffic-forecast')
os.makedirs(OUT, exist_ok=True)
!ls

## 3 · Install packages

In [ ]:
!pip install -q -r requirements.txt xgboost mlflow
import tensorflow as tf, xgboost
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))
print('XGBoost', xgboost.__version__)

## 4 · Build the processed dataset
Runs the same pipeline as notebook 02 and saves `data/processed/dataset.parquet`.

In [ ]:
import sys; sys.path.insert(0, '.')
from src.data import load_traffic, load_weather, load_fuel_events, ROOT
from src.processing import build_dataset
df, meta, _=build_dataset(load_traffic(), load_weather(), load_fuel_events())
(ROOT/'data'/'processed').mkdir(parents=True, exist_ok=True)
df.to_parquet(ROOT/'data'/'processed'/'dataset.parquet')
print(df.shape, meta)

## 5 · Smoke test (2–3 minutes)
A very short run to make sure everything works before the long runs.

In [ ]:
!python -m src.train --out $OUT --quick

## 6 · Final test stage (~1 hour)
Train Jan–Aug, early stopping on Sep–Oct, evaluate once on Nov–Dec.
3 horizons × (2 baselines + XGBoost and LSTM × 5 feature sets × 3 seeds).

In [ ]:
!python -m src.train --out $OUT --stage test

## 7 · Walk-forward stage (~1 hour)
Train on months 1…m, evaluate on month m+1, for May to October. Full model and model without neighbours.

In [ ]:
!python -m src.train --out $OUT --stage walkforward

## 8 · Pack the results
Creates `mahmutbey_results.zip` in your Drive. **Send this file back** (results.csv + predictions).

In [ ]:
import shutil, pandas as pd
r=pd.read_csv(f'{OUT}/results.csv')
print(len(r), 'runs finished')
display(r.query("stage=='test' and ablation=='full'").groupby(['horizon', 'model'])[['MAE', 'RMSE', 'MAPE', 'R2', 'PICP90', 'MPIW90']].mean().round(2))
shutil.make_archive('/content/drive/MyDrive/mahmutbey_results', 'zip', OUT)
print('Saved: MyDrive/mahmutbey_results.zip')